In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip -q "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/KaggleDataset/ISICDataset.zip" -d "/content/data/"
print("Extrated files")

Extrated files


In [ ]:
import os
import io
import json
import copy
import math
import random
import warnings
warnings.filterwarnings("ignore")

import h5py
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

from torchvision import models
from torchvision.models import (
    EfficientNet_B0_Weights,
    MobileNet_V3_Large_Weights,
)

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    matthews_corrcoef,
    cohen_kappa_score,
    balanced_accuracy_score,
    brier_score_loss,
)

In [ ]:
KAGGLE_INPUT = "/content/data"
TRAIN_HDF5 = f"{KAGGLE_INPUT}/train-image.hdf5"
INDEX_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/final_dataset_index.csv"

# Put your teacher checkpoint path here
TEACHER_MODEL_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/tech/efficientnet_b0_fusion_teacher_best.pth"

SAVE_DIR = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation"
BEST_MODEL_PATH = os.path.join(SAVE_DIR, "mobilenetv3_large_fusion_student_best.pth")
LAST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "checkpoint_last.pth")
METRICS_JSON_PATH = os.path.join(SAVE_DIR, "final_metrics.json")
HISTORY_CSV_PATH = os.path.join(SAVE_DIR, "training_history.csv")
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cpu


In [ ]:
# CONFIG
# ============================================================

CONFIG = {
    "img_size": 224,
    "batch_size": 32,
    "epochs": 25,
    "student_warmup_epochs": 2,

    "lr_backbone": 5e-5,
    "lr_head": 2e-4,
    "weight_decay": 1e-4,

    "patience": 6,
    "num_workers": 2,
    "pin_memory": True,
    "amp": True,

    # student architecture
    "dropout": 0.30,
    "meta_hidden": 64,
    "fusion_hidden": 256,

    # one disciplined imbalance strategy
    "use_weighted_sampler": True,
    "positive_boost_cap": 12.0,
    "epoch_num_samples": 80000,

    # KD settings
    "temperature": 3.0,
    "alpha_kd": 0.35,      # soft loss weight
    "alpha_hard": 0.65,    # hard BCE weight

    # evaluation
    "target_sensitivity": 0.80,

    # resume
    "resume_checkpoint": None,
}

In [ ]:
# METRICS
# ============================================================

def compute_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp + 1e-8)

def compute_npv(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fn + 1e-8)

def sensitivity_at_specificity(y_true, y_prob, target_specificity=0.95):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    specificity = 1.0 - fpr
    valid = np.where(specificity >= target_specificity)[0]
    if len(valid) == 0:
        return 0.0, 1.0
    idx = valid[-1]
    return float(tpr[idx]), float(thresholds[idx])

def threshold_at_sensitivity_max_precision(y_true, y_prob, min_sensitivity=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = np.append(thresholds, 1.0)

    valid = np.where(recall >= min_sensitivity)[0]
    if len(valid) == 0:
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        idx = int(np.argmax(f1))
        return float(thresholds[idx])

    valid_prec = precision[valid]
    best_prec = np.max(valid_prec)
    tied = valid[valid_prec == best_prec]
    chosen = tied[np.argmax(thresholds[tied])]
    return float(thresholds[chosen])

def threshold_best_f1(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = np.append(thresholds, 1.0)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    idx = int(np.argmax(f1))
    return float(thresholds[idx]), float(f1[idx])

def evaluate_predictions(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(np.float64)
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_sensitivity": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(compute_specificity(y_true, y_pred)),
        "npv": float(compute_npv(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "ap": float(average_precision_score(y_true, y_prob)),
        "sens@95spec": float(sensitivity_at_specificity(y_true, y_prob, 0.95)[0]),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }
    return metrics, cm

def print_metrics(title, metrics):
    print(f"\n{'=' * 90}")
    print(title)
    print(f"{'=' * 90}")
    for k, v in metrics.items():
        if k in ["tn", "fp", "fn", "tp"]:
            print(f"{k:>24}: {v}")
        else:
            print(f"{k:>24}: {v:.6f}")

In [ ]:
# METADATA PREPROCESSOR
# ============================================================

class MetadataPreprocessor:
    def __init__(self):
        self.age_median = None
        self.age_min = None
        self.age_max = None
        self.size_median = None
        self.size_q99 = None
        self.site_cols = None

    def fit(self, df):
        df = df.copy()

        self.age_median = float(df["age_approx"].median())
        age_filled = df["age_approx"].fillna(self.age_median)
        self.age_min = float(age_filled.min())
        self.age_max = float(age_filled.max())

        if "clin_size_long_diam_mm" in df.columns:
            size_series = df["clin_size_long_diam_mm"]
            if size_series.notna().any():
                self.size_median = float(size_series.median())
                self.size_q99 = float(size_series.quantile(0.99))
                if self.size_q99 <= 0:
                    self.size_q99 = 1.0
            else:
                self.size_median = 0.0
                self.size_q99 = 1.0
        else:
            self.size_median = 0.0
            self.size_q99 = 1.0

        site_series = df["anatom_site_general"].fillna("unknown").astype(str)
        self.site_cols = sorted(site_series.unique().tolist())
        return self

    def transform(self, df):
        df = df.copy()

        df["age_approx"] = df["age_approx"].fillna(self.age_median)
        df["age_approx"] = (df["age_approx"] - self.age_min) / (self.age_max - self.age_min + 1e-8)

        df["sex_missing"] = df["sex"].isna().astype(np.float32)
        df["sex_male"] = (df["sex"] == "male").astype(np.float32)
        df["sex_female"] = (df["sex"] == "female").astype(np.float32)

        if "clin_size_long_diam_mm" in df.columns:
            df["clin_size_long_diam_mm"] = df["clin_size_long_diam_mm"].fillna(self.size_median)
            df["clin_size_long_diam_mm"] = df["clin_size_long_diam_mm"].clip(0, self.size_q99) / (self.size_q99 + 1e-8)
        else:
            df["clin_size_long_diam_mm"] = 0.0

        site = df["anatom_site_general"].fillna("unknown").astype(str)
        for col in self.site_cols:
            df[f"site_{col}"] = (site == col).astype(np.float32)

        meta_cols = [
            "age_approx",
            "sex_male",
            "sex_female",
            "sex_missing",
            "clin_size_long_diam_mm",
        ] + [f"site_{c}" for c in self.site_cols]

        for c in meta_cols:
            if c not in df.columns:
                df[c] = 0.0

        return df, meta_cols

In [ ]:
# DATASET
# ============================================================

class ISICFusionDataset(Dataset):
    def __init__(self, df, hdf5_path, meta_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.hdf5_path = hdf5_path
        self.meta_cols = meta_cols
        self.transform = transform
        self._h5 = None

    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.hdf5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        isic_id = row["isic_id"]

        h5 = self._get_h5()
        raw = h5[isic_id][()]
        image = np.array(Image.open(io.BytesIO(raw)).convert("RGB"))

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        meta = torch.tensor(row[self.meta_cols].values.astype(np.float32), dtype=torch.float32)
        label = torch.tensor(float(row["target"]), dtype=torch.float32)
        return image, meta, label

In [ ]:
# AUGMENTATION
# ============================================================

train_tf = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=12, border_mode=cv2.BORDER_REFLECT_101, p=0.35),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.25),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=8, val_shift_limit=5, p=0.15),
    A.GaussNoise(std_range=(0.02, 0.08), mean_range=(0.0, 0.0), p=0.10),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
# TEACHER MODEL
# ============================================================

class EfficientNetB0Fusion(nn.Module):
    def __init__(self, meta_dim, dropout=0.30, meta_hidden=64, fusion_hidden=256):
        super().__init__()

        backbone = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.image_proj = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(dropout),
        )

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, meta_hidden),
            nn.BatchNorm1d(meta_hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(meta_hidden, 64),
            nn.SiLU(),
        )

        self.gate = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.SiLU(),
            nn.Linear(128, 256),
            nn.Sigmoid(),
        )

        self.meta_to_img = nn.Sequential(
            nn.Linear(64, 256),
            nn.SiLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, fusion_hidden),
            nn.BatchNorm1d(fusion_hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        img_feat = self.image_proj(img_feat)

        meta_feat = self.meta_net(meta)
        meta_as_img = self.meta_to_img(meta_feat)

        gate = self.gate(torch.cat([img_feat, meta_feat], dim=1))
        gated_img = gate * img_feat + (1.0 - gate) * meta_as_img

        fused = torch.cat([gated_img, meta_feat], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits

In [ ]:
# TEACHER MODEL
# ============================================================

class EfficientNetB0Fusion(nn.Module):
    def __init__(self, meta_dim, dropout=0.30, meta_hidden=64, fusion_hidden=256):
        super().__init__()

        backbone = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.image_proj = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(dropout),
        )

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, meta_hidden),
            nn.BatchNorm1d(meta_hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(meta_hidden, 64),
            nn.SiLU(),
        )

        self.gate = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.SiLU(),
            nn.Linear(128, 256),
            nn.Sigmoid(),
        )

        self.meta_to_img = nn.Sequential(
            nn.Linear(64, 256),
            nn.SiLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, fusion_hidden),
            nn.BatchNorm1d(fusion_hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        img_feat = self.image_proj(img_feat)

        meta_feat = self.meta_net(meta)
        meta_as_img = self.meta_to_img(meta_feat)

        gate = self.gate(torch.cat([img_feat, meta_feat], dim=1))
        gated_img = gate * img_feat + (1.0 - gate) * meta_as_img

        fused = torch.cat([gated_img, meta_feat], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits

In [ ]:
# STUDENT MODEL
# ============================================================

class MobileNetV3Fusion(nn.Module):
    def __init__(self, meta_dim, dropout=0.30, meta_hidden=64, fusion_hidden=256):
        super().__init__()

        backbone = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.image_proj = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(dropout),
        )

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, meta_hidden),
            nn.BatchNorm1d(meta_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(meta_hidden, 64),
            nn.ReLU(inplace=True),
        )

        self.gate = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 256),
            nn.Sigmoid(),
        )

        self.meta_to_img = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(inplace=True),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, fusion_hidden),
            nn.BatchNorm1d(fusion_hidden),
            nn.Hardswish(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        img_feat = self.image_proj(img_feat)

        meta_feat = self.meta_net(meta)
        meta_as_img = self.meta_to_img(meta_feat)

        gate = self.gate(torch.cat([img_feat, meta_feat], dim=1))
        gated_img = gate * img_feat + (1.0 - gate) * meta_as_img

        fused = torch.cat([gated_img, meta_feat], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits

def set_backbone_trainable(model, trainable):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

In [ ]:
# KD LOSS
# ============================================================

class BinaryDistillationLoss(nn.Module):
    def __init__(self, temperature=3.0, alpha_kd=0.35, alpha_hard=0.65):
        super().__init__()
        self.temperature = temperature
        self.alpha_kd = alpha_kd
        self.alpha_hard = alpha_hard
        self.hard_loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, student_logits, teacher_logits, targets):
        # hard loss
        hard_loss = self.hard_loss_fn(student_logits, targets)

        # soft loss (AMP-safe)
        T = self.temperature
        teacher_probs = torch.sigmoid(teacher_logits / T).detach()

        soft_loss = F.binary_cross_entropy_with_logits(
            student_logits / T,
            teacher_probs
        )

        total_loss = self.alpha_hard * hard_loss + self.alpha_kd * (T * T) * soft_loss
        return total_loss, hard_loss.detach(), soft_loss.detach()

In [ ]:
# CHECKPOINT HELPERS
# ============================================================

def save_checkpoint(path, epoch, student_model, optimizer, scheduler, scaler, best_val_ap, history, meta_cols, preprocessor_state):
    state = {
        "epoch": epoch,
        "model_state": student_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict(),
        "best_val_ap": best_val_ap,
        "history": history,
        "meta_cols": meta_cols,
        "preprocessor_state": preprocessor_state,
        "config": CONFIG,
        "seed": SEED,
    }
    torch.save(state, path)

def load_checkpoint(path, student_model, optimizer=None, scheduler=None, scaler=None):
    ckpt = torch.load(path, map_location=DEVICE)
    student_model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler is not None and "scaler_state" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state"])
    return ckpt

In [ ]:
# TRAIN / EVAL
# ============================================================

def run_epoch_train_kd(student_model, teacher_model, loader, optimizer, kd_criterion, scaler):
    student_model.train()
    teacher_model.eval()

    running_loss = 0.0
    running_hard = 0.0
    running_soft = 0.0

    pbar = tqdm(loader, desc="Train KD", leave=False)
    for images, meta, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        meta = meta.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            teacher_logits = teacher_model(images, meta)

        with autocast(enabled=CONFIG["amp"] and DEVICE.type == "cuda"):
            student_logits = student_model(images, meta)
            loss, hard_loss, soft_loss = kd_criterion(student_logits, teacher_logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        running_hard += hard_loss.item() * images.size(0)
        running_soft += soft_loss.item() * images.size(0)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            hard=f"{hard_loss.item():.4f}",
            soft=f"{soft_loss.item():.4f}"
        )

    n = len(loader.dataset)
    return running_loss / n, running_hard / n, running_soft / n

@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    all_logits = []
    all_labels = []

    pbar = tqdm(loader, desc="Eval", leave=False)
    for images, meta, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        meta = meta.to(DEVICE, non_blocking=True)

        with autocast(enabled=CONFIG["amp"] and DEVICE.type == "cuda"):
            logits = model(images, meta)

        all_logits.append(logits.detach().float().cpu().numpy())
        all_labels.append(labels.numpy())

    all_logits = np.concatenate(all_logits).astype(np.float64)
    all_labels = np.concatenate(all_labels).astype(int)
    return all_labels, all_logits

def logits_to_probs(logits):
    logits = np.asarray(logits, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-logits))

In [ ]:
# LOAD DATA
# ============================================================

assert os.path.exists(INDEX_PATH), f"Missing INDEX_PATH: {INDEX_PATH}"
assert os.path.exists(TRAIN_HDF5), f"Missing TRAIN_HDF5: {TRAIN_HDF5}"
assert os.path.exists(TEACHER_MODEL_PATH), f"Missing TEACHER_MODEL_PATH: {TEACHER_MODEL_PATH}"

df = pd.read_csv(INDEX_PATH)
print("Full dataframe shape:", df.shape)

required_cols = ["isic_id", "target", "split", "age_approx", "sex", "anatom_site_general"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Required column missing: {c}")

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")
print("Train malignant:", int(train_df["target"].sum()))
print("Train benign:", int((train_df["target"] == 0).sum()))

preprocessor = MetadataPreprocessor().fit(train_df)
train_df, meta_cols = preprocessor.transform(train_df)
val_df, _ = preprocessor.transform(val_df)
test_df, _ = preprocessor.transform(test_df)

preprocessor_state = {
    "age_median": preprocessor.age_median,
    "age_min": preprocessor.age_min,
    "age_max": preprocessor.age_max,
    "size_median": preprocessor.size_median,
    "size_q99": preprocessor.size_q99,
    "site_cols": preprocessor.site_cols,
}

print("Meta dim:", len(meta_cols))
print("Metadata features:", meta_cols)

train_dataset = ISICFusionDataset(train_df, TRAIN_HDF5, meta_cols, transform=train_tf)
val_dataset = ISICFusionDataset(val_df, TRAIN_HDF5, meta_cols, transform=val_tf)
test_dataset = ISICFusionDataset(test_df, TRAIN_HDF5, meta_cols, transform=val_tf)

loader_kwargs = dict(
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
    persistent_workers=CONFIG["num_workers"] > 0,
)

if CONFIG["use_weighted_sampler"]:
    num_pos = int(train_df["target"].sum())
    num_neg = int((train_df["target"] == 0).sum())
    raw_ratio = num_neg / max(num_pos, 1)
    pos_boost = min(math.sqrt(raw_ratio), CONFIG["positive_boost_cap"])

    sample_weights = np.where(train_df["target"].values == 1, pos_boost, 1.0).astype(np.float64)
    num_samples = min(CONFIG["epoch_num_samples"], len(train_df))

    train_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
    )

    train_loader = DataLoader(
        train_dataset,
        sampler=train_sampler,
        shuffle=False,
        **loader_kwargs,
    )
    print(f"Using weighted sampler | pos_boost={pos_boost:.3f} | epoch_samples={num_samples}")
else:
    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        **loader_kwargs,
    )

val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

Full dataframe shape: (401059, 8)
Train size: 298560
Val size:   53629
Test size:  48870
Train malignant: 271
Train benign: 298289
Meta dim: 11
Metadata features: ['age_approx', 'sex_male', 'sex_female', 'sex_missing', 'clin_size_long_diam_mm', 'site_anterior torso', 'site_head/neck', 'site_lower extremity', 'site_posterior torso', 'site_unknown', 'site_upper extremity']
Using weighted sampler | pos_boost=12.000 | epoch_samples=80000


In [ ]:
# BUILD TEACHER + STUDENT
# ============================================================

teacher_model = EfficientNetB0Fusion(
    meta_dim=len(meta_cols),
    dropout=0.30,
    meta_hidden=64,
    fusion_hidden=256,
).to(DEVICE)

teacher_state = torch.load(TEACHER_MODEL_PATH, map_location=DEVICE)
teacher_model.load_state_dict(teacher_state)
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

student_model = MobileNetV3Fusion(
    meta_dim=len(meta_cols),
    dropout=CONFIG["dropout"],
    meta_hidden=CONFIG["meta_hidden"],
    fusion_hidden=CONFIG["fusion_hidden"],
).to(DEVICE)

optimizer = optim.AdamW([
    {"params": student_model.backbone.parameters(), "lr": CONFIG["lr_backbone"]},
    {
        "params": list(student_model.image_proj.parameters()) +
                  list(student_model.meta_net.parameters()) +
                  list(student_model.gate.parameters()) +
                  list(student_model.meta_to_img.parameters()) +
                  list(student_model.classifier.parameters()),
        "lr": CONFIG["lr_head"],
    }
], weight_decay=CONFIG["weight_decay"])

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=1e-6,
)

scaler = GradScaler(enabled=CONFIG["amp"] and DEVICE.type == "cuda")
kd_criterion = BinaryDistillationLoss(
    temperature=CONFIG["temperature"],
    alpha_kd=CONFIG["alpha_kd"],
    alpha_hard=CONFIG["alpha_hard"],
)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 93.2MB/s]


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 95.3MB/s]


In [ ]:
# RESUME SUPPORT
# ============================================================

start_epoch = 1
best_val_ap = -np.inf
history = []

resume_path = CONFIG["resume_checkpoint"]
if resume_path is not None and os.path.exists(resume_path):
    print(f"Resuming from: {resume_path}")
    ckpt = load_checkpoint(resume_path, student_model, optimizer, scheduler, scaler)
    start_epoch = ckpt["epoch"] + 1
    best_val_ap = ckpt.get("best_val_ap", -np.inf)
    history = ckpt.get("history", [])
    print(f"Resumed at epoch {start_epoch}")

In [ ]:
# TRAINING LOOP
# ============================================================

best_state = None
epochs_no_improve = 0

for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['epochs']}")

    if epoch <= CONFIG["student_warmup_epochs"]:
        set_backbone_trainable(student_model, False)
        print("Student backbone frozen.")
    else:
        set_backbone_trainable(student_model, True)
        print("Student backbone unfrozen.")

    train_loss, train_hard, train_soft = run_epoch_train_kd(
        student_model, teacher_model, train_loader, optimizer, kd_criterion, scaler
    )

    y_val, val_logits = collect_logits(student_model, val_loader)
    p_val = logits_to_probs(val_logits)

    val_threshold_sens = threshold_at_sensitivity_max_precision(
        y_val, p_val, CONFIG["target_sensitivity"]
    )
    val_metrics_sens, _ = evaluate_predictions(y_val, p_val, val_threshold_sens)

    val_threshold_f1, _ = threshold_best_f1(y_val, p_val)
    val_metrics_f1, _ = evaluate_predictions(y_val, p_val, val_threshold_f1)

    current_val_ap = val_metrics_sens["ap"]
    current_val_auroc = val_metrics_sens["auroc"]

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_hard_loss": train_hard,
        "train_soft_loss": train_soft,
        "val_ap": current_val_ap,
        "val_auroc": current_val_auroc,
        "val_sens95spec": val_metrics_sens["sens@95spec"],
        "val_sensitivity_thr": val_threshold_sens,
        "val_sensitivity": val_metrics_sens["recall_sensitivity"],
        "val_specificity": val_metrics_sens["specificity"],
        "val_precision": val_metrics_sens["precision"],
        "val_f1_at_sens_thr": val_metrics_sens["f1"],
        "val_best_f1_thr": val_threshold_f1,
        "val_best_f1": val_metrics_f1["f1"],
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_head": optimizer.param_groups[1]["lr"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)

    print(f"Train total loss: {train_loss:.6f}")
    print(f"Train hard loss:  {train_hard:.6f}")
    print(f"Train soft loss:  {train_soft:.6f}")

    print_metrics("Validation @ sensitivity-constrained threshold", val_metrics_sens)
    print_metrics("Validation @ best-F1 threshold", val_metrics_f1)

    save_checkpoint(
        LAST_CHECKPOINT_PATH, epoch, student_model, optimizer, scheduler, scaler,
        best_val_ap, history, meta_cols, preprocessor_state
    )

    epoch_ckpt = os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch:02d}.pth")
    save_checkpoint(
        epoch_ckpt, epoch, student_model, optimizer, scheduler, scaler,
        best_val_ap, history, meta_cols, preprocessor_state
    )

    if current_val_ap > best_val_ap:
        best_val_ap = current_val_ap
        best_state = copy.deepcopy(student_model.state_dict())
        torch.save(best_state, BEST_MODEL_PATH)
        print(f"New best student saved by validation AP: {best_val_ap:.6f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"No validation AP improvement for {epochs_no_improve} epoch(s)")

    if epochs_no_improve >= CONFIG["patience"]:
        print("Early stopping triggered.")
        break


Epoch 1/25
Student backbone frozen.


Train KD:   0%|          | 6/2500 [00:29<3:01:39,  4.37s/it, hard=0.6481, loss=2.5643, soft=0.6803]

In [ ]:
# TRAINING LOOP
# ============================================================

best_state = None
epochs_no_improve = 0

for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['epochs']}")

    if epoch <= CONFIG["student_warmup_epochs"]:
        set_backbone_trainable(student_model, False)
        print("Student backbone frozen.")
    else:
        set_backbone_trainable(student_model, True)
        print("Student backbone unfrozen.")

    train_loss, train_hard, train_soft = run_epoch_train_kd(
        student_model, teacher_model, train_loader, optimizer, kd_criterion, scaler
    )

    y_val, val_logits = collect_logits(student_model, val_loader)
    p_val = logits_to_probs(val_logits)

    val_threshold_sens = threshold_at_sensitivity_max_precision(
        y_val, p_val, CONFIG["target_sensitivity"]
    )
    val_metrics_sens, _ = evaluate_predictions(y_val, p_val, val_threshold_sens)

    val_threshold_f1, _ = threshold_best_f1(y_val, p_val)
    val_metrics_f1, _ = evaluate_predictions(y_val, p_val, val_threshold_f1)

    current_val_ap = val_metrics_sens["ap"]
    current_val_auroc = val_metrics_sens["auroc"]

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_hard_loss": train_hard,
        "train_soft_loss": train_soft,
        "val_ap": current_val_ap,
        "val_auroc": current_val_auroc,
        "val_sens95spec": val_metrics_sens["sens@95spec"],
        "val_sensitivity_thr": val_threshold_sens,
        "val_sensitivity": val_metrics_sens["recall_sensitivity"],
        "val_specificity": val_metrics_sens["specificity"],
        "val_precision": val_metrics_sens["precision"],
        "val_f1_at_sens_thr": val_metrics_sens["f1"],
        "val_best_f1_thr": val_threshold_f1,
        "val_best_f1": val_metrics_f1["f1"],
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_head": optimizer.param_groups[1]["lr"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)

    print(f"Train total loss: {train_loss:.6f}")
    print(f"Train hard loss:  {train_hard:.6f}")
    print(f"Train soft loss:  {train_soft:.6f}")

    print_metrics("Validation @ sensitivity-constrained threshold", val_metrics_sens)
    print_metrics("Validation @ best-F1 threshold", val_metrics_f1)

    save_checkpoint(
        LAST_CHECKPOINT_PATH, epoch, student_model, optimizer, scheduler, scaler,
        best_val_ap, history, meta_cols, preprocessor_state
    )

    epoch_ckpt = os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch:02d}.pth")
    save_checkpoint(
        epoch_ckpt, epoch, student_model, optimizer, scheduler, scaler,
        best_val_ap, history, meta_cols, preprocessor_state
    )

    if current_val_ap > best_val_ap:
        best_val_ap = current_val_ap
        best_state = copy.deepcopy(student_model.state_dict())
        torch.save(best_state, BEST_MODEL_PATH)
        print(f"New best student saved by validation AP: {best_val_ap:.6f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"No validation AP improvement for {epochs_no_improve} epoch(s)")

    if epochs_no_improve >= CONFIG["patience"]:
        print("Early stopping triggered.")
        break

In [ ]:
# LOAD BEST STUDENT

assert os.path.exists(BEST_MODEL_PATH), "Best student model not found."
student_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
print(f"Loaded best student model from: {BEST_MODEL_PATH}")

In [ ]:
final_output = {
    "validation_at_sensitivity_threshold": val_metrics_sens,
    "test_at_sensitivity_threshold": test_metrics_sens,
    "validation_at_best_f1_threshold": val_metrics_f1,
    "test_at_best_f1_threshold": test_metrics_f1,
    "history": history,
    "teacher_model_path": TEACHER_MODEL_PATH,
}

with open(METRICS_JSON_PATH, "w") as f:
    json.dump(final_output, f, indent=2)

In [ ]:
# PLOTS
# ============================================================

history_df = pd.DataFrame(history)

if len(history_df) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Total Loss")
    plt.plot(history_df["epoch"], history_df["train_hard_loss"], label="Train Hard Loss")
    plt.plot(history_df["epoch"], history_df["train_soft_loss"], label="Train Soft Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("KD Training Losses")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "kd_losses.png"), dpi=150)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["val_auroc"], label="Val AUROC")
    plt.plot(history_df["epoch"], history_df["val_ap"], label="Val AP")
    plt.plot(history_df["epoch"], history_df["val_sens95spec"], label="Val Sens@95Spec")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Student Validation Metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "val_metrics_curve.png"), dpi=150)
    plt.show()

def plot_cm(cm, title, save_name):
    cm_norm = cm.astype(np.float64) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        xticklabels=["Pred Benign", "Pred Malignant"],
        yticklabels=["True Benign", "True Malignant"],
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, save_name), dpi=150)
    plt.show()

plot_cm(test_cm_sens, "Student Test CM (Sensitivity-Constrained)", "cm_test_sensitivity_threshold.png")
plot_cm(test_cm_f1, "Student Test CM (Best-F1)", "cm_test_best_f1_threshold.png")

print("\nKnowledge distillation training finished.")
print(f"Best student: {BEST_MODEL_PATH}")
print(f"Checkpoint:   {LAST_CHECKPOINT_PATH}")
print(f"Metrics JSON: {METRICS_JSON_PATH}")
print(f"History CSV:  {HISTORY_CSV_PATH}")

In [6]:
# ============================================================
# XAI PIPELINE FOR MOBILE NET V3 LARGE FUSION STUDENT
# - Grad-CAM for binary classification
# - Saves TP / FP / FN / TN examples
# - Saves overlays + summary CSV
# - Kaggle-ready
# ============================================================

import os
import io
import math
import json
import random
import warnings
warnings.filterwarnings("ignore")

import h5py
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

KAGGLE_INPUT = "/content/data"
TRAIN_HDF5 = f"{KAGGLE_INPUT}/train-image.hdf5"

# student checkpoint from KD
MODEL_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/mobilenetv3_large_fusion_student_best.pth"

SAVE_DIR = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/xai_student_mobilenetv3"
INDEX_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/final_dataset_index.csv"
HDF5_PATH = f"{KAGGLE_INPUT}/train-image.hdf5"
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# how many examples to save per category
MAX_PER_GROUP = 20

# choose threshold mode
THRESHOLD_MODE = "best_f1"   # "best_f1" or "target_sensitivity"
TARGET_SENSITIVITY = 0.80

# consistency test settings
RUN_CONSISTENCY_CHECK = True
CONSISTENCY_SAMPLES_PER_GROUP = 5

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# METADATA PREPROCESSOR
# ============================================================

class MetadataPreprocessor:
    def __init__(self):
        self.age_median = None
        self.age_min = None
        self.age_max = None
        self.size_median = None
        self.size_q99 = None
        self.site_cols = None

    def fit(self, df):
        df = df.copy()

        self.age_median = float(df["age_approx"].median())
        age_filled = df["age_approx"].fillna(self.age_median)
        self.age_min = float(age_filled.min())
        self.age_max = float(age_filled.max())

        if "clin_size_long_diam_mm" in df.columns:
            size_series = df["clin_size_long_diam_mm"]
            if size_series.notna().any():
                self.size_median = float(size_series.median())
                self.size_q99 = float(size_series.quantile(0.99))
                if self.size_q99 <= 0:
                    self.size_q99 = 1.0
            else:
                self.size_median = 0.0
                self.size_q99 = 1.0
        else:
            self.size_median = 0.0
            self.size_q99 = 1.0

        site_series = df["anatom_site_general"].fillna("unknown").astype(str)
        self.site_cols = sorted(site_series.unique().tolist())
        return self

    def transform(self, df):
        df = df.copy()

        df["age_approx"] = df["age_approx"].fillna(self.age_median)
        df["age_approx"] = (df["age_approx"] - self.age_min) / (self.age_max - self.age_min + 1e-8)

        df["sex_missing"] = df["sex"].isna().astype(np.float32)
        df["sex_male"] = (df["sex"] == "male").astype(np.float32)
        df["sex_female"] = (df["sex"] == "female").astype(np.float32)

        if "clin_size_long_diam_mm" in df.columns:
            df["clin_size_long_diam_mm"] = df["clin_size_long_diam_mm"].fillna(self.size_median)
            df["clin_size_long_diam_mm"] = df["clin_size_long_diam_mm"].clip(0, self.size_q99) / (self.size_q99 + 1e-8)
        else:
            df["clin_size_long_diam_mm"] = 0.0

        site = df["anatom_site_general"].fillna("unknown").astype(str)
        for col in self.site_cols:
            df[f"site_{col}"] = (site == col).astype(np.float32)

        meta_cols = [
            "age_approx",
            "sex_male",
            "sex_female",
            "sex_missing",
            "clin_size_long_diam_mm",
        ] + [f"site_{c}" for c in self.site_cols]

        for c in meta_cols:
            if c not in df.columns:
                df[c] = 0.0

        return df, meta_cols

# ============================================================
# DATASET
# ============================================================

class ISICFusionDataset(Dataset):
    def __init__(self, df, hdf5_path, meta_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.hdf5_path = hdf5_path
        self.meta_cols = meta_cols
        self.transform = transform
        self._h5 = None

    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.hdf5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        isic_id = row["isic_id"]

        h5 = self._get_h5()
        raw = h5[isic_id][()]
        image = np.array(Image.open(io.BytesIO(raw)).convert("RGB"))

        image_for_save = image.copy()

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        meta = torch.tensor(row[self.meta_cols].values.astype(np.float32), dtype=torch.float32)
        label = torch.tensor(float(row["target"]), dtype=torch.float32)

        return image, meta, label, isic_id, image_for_save

# ============================================================
# MODEL
# ============================================================

class MobileNetV3Fusion(nn.Module):
    def __init__(self, meta_dim, dropout=0.30, meta_hidden=64, fusion_hidden=256):
        super().__init__()

        backbone = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.image_proj = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(dropout),
        )

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, meta_hidden),
            nn.BatchNorm1d(meta_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(meta_hidden, 64),
            nn.ReLU(inplace=True),
        )

        self.gate = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 256),
            nn.Sigmoid(),
        )

        self.meta_to_img = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(inplace=True),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, fusion_hidden),
            nn.BatchNorm1d(fusion_hidden),
            nn.Hardswish(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        img_feat = self.image_proj(img_feat)

        meta_feat = self.meta_net(meta)
        meta_as_img = self.meta_to_img(meta_feat)

        gate = self.gate(torch.cat([img_feat, meta_feat], dim=1))
        gated_img = gate * img_feat + (1.0 - gate) * meta_as_img

        fused = torch.cat([gated_img, meta_feat], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits

# ============================================================
# GRAD-CAM FOR BINARY MODEL
# ============================================================

class GradCAMBinary:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.fwd_hook = target_layer.register_forward_hook(self._save_activations)
        self.bwd_hook = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.fwd_hook.remove()
        self.bwd_hook.remove()

    def generate(self, image_tensor, meta_tensor):
        self.model.eval()
        self.model.zero_grad()

        logit = self.model(image_tensor, meta_tensor)   # [1]
        score = logit[0]
        score.backward(retain_graph=True)

        activations = self.activations[0]   # [C,H,W]
        gradients = self.gradients[0]       # [C,H,W]

        weights = gradients.mean(dim=(1, 2), keepdim=True)
        cam = (weights * activations).sum(dim=0)
        cam = torch.relu(cam)

        cam -= cam.min()
        cam /= (cam.max() + 1e-8)

        prob = torch.sigmoid(logit)[0].item()
        return cam.detach().cpu().numpy(), prob, score.item()

# ============================================================
# TRANSFORMS
# ============================================================

infer_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

consistency_tf = [
    A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]),
    A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]),
    A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.RandomBrightnessContrast(brightness_limit=0.05, contrast_limit=0.05, p=1.0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]),
]

# ============================================================
# HELPERS
# ============================================================

def collate_fn(batch):
    images, metas, labels, isic_ids, raw_images = zip(*batch)
    images = torch.stack(images, dim=0)
    metas = torch.stack(metas, dim=0)
    labels = torch.stack(labels, dim=0)
    return images, metas, labels, list(isic_ids), list(raw_images)

def threshold_at_sensitivity_max_precision(y_true, y_prob, min_sensitivity=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = np.append(thresholds, 1.0)

    valid = np.where(recall >= min_sensitivity)[0]
    if len(valid) == 0:
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        idx = int(np.argmax(f1))
        return float(thresholds[idx])

    valid_prec = precision[valid]
    best_prec = np.max(valid_prec)
    tied = valid[valid_prec == best_prec]
    chosen = tied[np.argmax(thresholds[tied])]
    return float(thresholds[chosen])

def threshold_best_f1(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = np.append(thresholds, 1.0)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    idx = int(np.argmax(f1))
    return float(thresholds[idx])

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_rgb(path, img):
    Image.fromarray(img.astype(np.uint8)).save(path)

def make_overlay(original_img, cam):
    cam_resized = cv2.resize(cam, (original_img.shape[1], original_img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.4 * heatmap + 0.6 * original_img).astype(np.uint8)
    return cam_resized, heatmap, overlay

def save_cam_panel(save_path, original_img, cam_resized, heatmap, overlay, title_text):
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(original_img)
    axes[0].set_title("Original")
    axes[1].imshow(cam_resized, cmap="jet")
    axes[1].set_title("Grad-CAM")
    axes[2].imshow(heatmap)
    axes[2].set_title("Heatmap")
    axes[3].imshow(overlay)
    axes[3].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(title_text)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def compute_iou(cam_a, cam_b, thr=0.5):
    a = (cam_a >= thr).astype(np.uint8)
    b = (cam_b >= thr).astype(np.uint8)
    inter = (a & b).sum()
    union = (a | b).sum()
    if union == 0:
        return 0.0
    return float(inter / union)

# ============================================================
# LOAD DATA
# ============================================================

assert os.path.exists(HDF5_PATH), f"Missing HDF5_PATH: {HDF5_PATH}"
assert os.path.exists(INDEX_PATH), f"Missing INDEX_PATH: {INDEX_PATH}"
assert os.path.exists(MODEL_PATH), f"Missing MODEL_PATH: {MODEL_PATH}"

df = pd.read_csv(INDEX_PATH)

train_df_raw = df[df["split"] == "train"].copy()
val_df_raw = df[df["split"] == "val"].copy()
test_df_raw = df[df["split"] == "test"].copy()

preprocessor = MetadataPreprocessor().fit(train_df_raw)
train_df, meta_cols = preprocessor.transform(train_df_raw)
val_df, _ = preprocessor.transform(val_df_raw)
test_df, _ = preprocessor.transform(test_df_raw)

print("Meta dim:", len(meta_cols))

val_ds = ISICFusionDataset(val_df, HDF5_PATH, meta_cols, transform=infer_tf)
test_ds = ISICFusionDataset(test_df, HDF5_PATH, meta_cols, transform=infer_tf)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_fn
)

# ============================================================
# LOAD MODEL
# ============================================================

model = MobileNetV3Fusion(meta_dim=len(meta_cols)).to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

# good target layer for MobileNetV3-Large
target_layer = model.backbone.features[-1]
gradcam = GradCAMBinary(model, target_layer)

# ============================================================
# COLLECT VAL PROBS FOR THRESHOLD
# ============================================================

@torch.no_grad()
def collect_probs(loader):
    all_probs = []
    all_labels = []
    all_ids = []

    for images, metas, labels, isic_ids, raw_images in tqdm(loader, desc="Collect probs", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        metas = metas.to(DEVICE, non_blocking=True)

        logits = model(images, metas)
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_ids.extend(isic_ids)

    return np.array(all_labels).astype(int), np.array(all_probs).astype(float), all_ids

y_val, p_val, _ = collect_probs(val_loader)

if THRESHOLD_MODE == "best_f1":
    threshold = threshold_best_f1(y_val, p_val)
elif THRESHOLD_MODE == "target_sensitivity":
    threshold = threshold_at_sensitivity_max_precision(y_val, p_val, TARGET_SENSITIVITY)
else:
    raise ValueError("THRESHOLD_MODE must be 'best_f1' or 'target_sensitivity'")

print(f"Chosen threshold ({THRESHOLD_MODE}): {threshold:.6f}")

# ============================================================
# RUN TEST + SAVE TP/FP/FN/TN EXAMPLES
# ============================================================

groups = {
    "TP": [],
    "FP": [],
    "FN": [],
    "TN": [],
}

for group_name in groups:
    ensure_dir(os.path.join(SAVE_DIR, group_name))
    ensure_dir(os.path.join(SAVE_DIR, group_name, "original"))
    ensure_dir(os.path.join(SAVE_DIR, group_name, "overlay"))
    ensure_dir(os.path.join(SAVE_DIR, group_name, "panel"))

summary_rows = []

# iterate one-by-one for Grad-CAM
for idx in tqdm(range(len(test_ds)), desc="Generating XAI"):
    image_t, meta_t, label_t, isic_id, raw_img = test_ds[idx]

    image_t = image_t.unsqueeze(0).to(DEVICE)
    meta_t = meta_t.unsqueeze(0).to(DEVICE)
    true_label = int(label_t.item())

    cam, prob, logit = gradcam.generate(image_t, meta_t)
    pred_label = int(prob >= threshold)

    if true_label == 1 and pred_label == 1:
        group = "TP"
    elif true_label == 0 and pred_label == 1:
        group = "FP"
    elif true_label == 1 and pred_label == 0:
        group = "FN"
    else:
        group = "TN"

    cam_resized, heatmap, overlay = make_overlay(raw_img, cam)

    summary_rows.append({
        "isic_id": isic_id,
        "true_label": true_label,
        "pred_label": pred_label,
        "probability": float(prob),
        "logit": float(logit),
        "group": group,
    })

    if len(groups[group]) < MAX_PER_GROUP:
        fname = f"{len(groups[group]):02d}_{isic_id}_p{prob:.4f}.png"

        save_rgb(os.path.join(SAVE_DIR, group, "original", fname), raw_img)
        save_rgb(os.path.join(SAVE_DIR, group, "overlay", fname), overlay)

        title = (
            f"{group} | ID={isic_id} | True={true_label} | Pred={pred_label} | "
            f"Prob={prob:.4f} | Thr={threshold:.4f}"
        )
        save_cam_panel(
            os.path.join(SAVE_DIR, group, "panel", fname),
            raw_img, cam_resized, heatmap, overlay, title
        )

        groups[group].append({
            "isic_id": isic_id,
            "probability": float(prob),
            "true_label": true_label,
            "pred_label": pred_label,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(SAVE_DIR, "xai_summary.csv"), index=False)

# ============================================================
# SAVE GROUP COUNTS
# ============================================================

group_counts = summary_df["group"].value_counts().to_dict()
with open(os.path.join(SAVE_DIR, "group_counts.json"), "w") as f:
    json.dump(group_counts, f, indent=2)

print("\nSaved examples per group:")
for g in ["TP", "FP", "FN", "TN"]:
    print(g, len(groups[g]))

# ============================================================
# CONSISTENCY CHECK
# ============================================================

def run_consistency_for_sample(isic_id, row, group_name, sample_idx):
    original_img = raw_img_from_hdf5(HDF5_PATH, isic_id)

    cams = []
    probs = []

    for aug_idx, tf in enumerate(consistency_tf):
        img_t = tf(image=original_img)["image"].unsqueeze(0).to(DEVICE)
        meta_t = torch.tensor(row[meta_cols].values.astype(np.float32), dtype=torch.float32).unsqueeze(0).to(DEVICE)

        cam, prob, _ = gradcam.generate(img_t, meta_t)
        cams.append(cam)
        probs.append(prob)

        cam_resized, heatmap, overlay = make_overlay(original_img, cam)
        save_path = os.path.join(
            SAVE_DIR, group_name, f"consistency_{sample_idx:02d}_{isic_id}_aug{aug_idx}.png"
        )
        title = f"{group_name} | {isic_id} | aug={aug_idx} | prob={prob:.4f}"
        save_cam_panel(save_path, original_img, cam_resized, heatmap, overlay, title)

    iou_01 = compute_iou(cams[0], cams[1], thr=0.5)
    iou_02 = compute_iou(cams[0], cams[2], thr=0.5)
    iou_12 = compute_iou(cams[1], cams[2], thr=0.5)

    return {
        "isic_id": isic_id,
        "group": group_name,
        "prob_orig": probs[0],
        "prob_aug1": probs[1],
        "prob_aug2": probs[2],
        "iou_orig_aug1": iou_01,
        "iou_orig_aug2": iou_02,
        "iou_aug1_aug2": iou_12,
        "mean_iou": float((iou_01 + iou_02 + iou_12) / 3.0),
    }

def raw_img_from_hdf5(hdf5_path, isic_id):
    with h5py.File(hdf5_path, "r") as f:
        raw = f[isic_id][()]
    image = np.array(Image.open(io.BytesIO(raw)).convert("RGB"))
    return image

if RUN_CONSISTENCY_CHECK:
    consistency_rows = []

    for group_name in ["TP", "FP", "FN", "TN"]:
        subset = summary_df[summary_df["group"] == group_name].head(CONSISTENCY_SAMPLES_PER_GROUP)
        for sample_idx, (_, rec) in enumerate(subset.iterrows()):
            isic_id = rec["isic_id"]
            row = test_df[test_df["isic_id"] == isic_id].iloc[0]
            result = run_consistency_for_sample(isic_id, row, group_name, sample_idx)
            consistency_rows.append(result)

    consistency_df = pd.DataFrame(consistency_rows)
    consistency_df.to_csv(os.path.join(SAVE_DIR, "consistency_check.csv"), index=False)

    if len(consistency_df) > 0:
        plt.figure(figsize=(8, 5))
        consistency_df.boxplot(column="mean_iou", by="group")
        plt.title("Grad-CAM Consistency by Group")
        plt.suptitle("")
        plt.ylabel("Mean IoU")
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, "consistency_boxplot.png"), dpi=200)
        plt.close()

# ============================================================
# CREATE CONTACT SHEETS
# ============================================================

def create_contact_sheet(image_paths, save_path, cols=4, thumb_size=(300, 300), title=None):
    if len(image_paths) == 0:
        return

    rows = math.ceil(len(image_paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    elif cols == 1:
        axes = np.array([[ax] for ax in axes])

    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, path in zip(axes, image_paths):
        img = np.array(Image.open(path).convert("RGB"))
        ax.imshow(img)
        ax.set_title(os.path.basename(path)[:40], fontsize=8)

    if title:
        fig.suptitle(title, fontsize=14)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

for group_name in ["TP", "FP", "FN", "TN"]:
    panel_dir = os.path.join(SAVE_DIR, group_name, "panel")
    paths = sorted([os.path.join(panel_dir, x) for x in os.listdir(panel_dir) if x.endswith(".png")])
    create_contact_sheet(
        paths,
        os.path.join(SAVE_DIR, f"{group_name}_contact_sheet.png"),
        cols=2,
        title=f"{group_name} Grad-CAM Examples"
    )

# ============================================================
# DONE
# ============================================================

gradcam.remove_hooks()

print("\nXAI pipeline finished.")
print("Outputs saved in:", SAVE_DIR)
print("Main files:")
print("-", os.path.join(SAVE_DIR, "xai_summary.csv"))
print("-", os.path.join(SAVE_DIR, "group_counts.json"))
if RUN_CONSISTENCY_CHECK:
    print("-", os.path.join(SAVE_DIR, "consistency_check.csv"))

Meta dim: 11
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 87.3MB/s]


Chosen threshold (best_f1): 0.825882


Generating XAI: 100%|██████████| 48870/48870 [1:53:43<00:00,  7.16it/s]



Saved examples per group:
TP 2
FP 15
FN 20
TN 20

XAI pipeline finished.
Outputs saved in: /content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/xai_student_mobilenetv3
Main files:
- /content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/xai_student_mobilenetv3/xai_summary.csv
- /content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/xai_student_mobilenetv3/group_counts.json
- /content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Teacher_Student_distillation/xai_student_mobilenetv3/consistency_check.csv


<Figure size 800x500 with 0 Axes>